# Credit Card Fraud Detection using MLP
This notebook implements a simplified version of the [Three-Model Fraud Detection System](https://medium.com/@nafisaidris413/building-a-three-model-real-time-fraud-detection-system-with-fastapi-d5b5c2de5544). We focus on a **Multilayer Perceptron (MLP)** to classify transactions as legitimate or fraudulent using the Kaggle Credit Card dataset.

### 1. Library Imports and Data Loading
We import essential libraries for deep learning (**PyTorch**), data manipulation (**Pandas**), and evaluation (**Scikit-Learn**). 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load the dataset
# Ensure creditcard.csv is in the same folder as this notebook
df = pd.read_csv('creditcard.csv')

### 2. Feature Scaling and Data Splitting
Neural networks perform best when numerical data is on a similar scale. We scale the 'Amount' and 'Time' columns and split the data into **Training (80%)** and **Testing (20%)** sets, ensuring we maintain the fraud-to-normal ratio using `stratify`.

In [ ]:
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['Time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

X = df.drop('Class', axis=1).values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert data to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

### 3. Defining the MLP Architecture
The MLP consists of an input layer, two hidden layers with **ReLU** activation functions, and a **Dropout** layer to prevent overfitting. The final layer outputs two values corresponding to the 'Normal' and 'Fraud' classes.

In [ ]:
class SimpleFraudMLP(nn.Module):
    def __init__(self, input_dim):
        super(SimpleFraudMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        
    def forward(self, x):
        return self.network(x)

model = SimpleFraudMLP(X_train.shape[1])

### 4. Training the Model
We use **CrossEntropyLoss** for classification and the **Adam optimizer**. We loop through the data for 10 epochs to update the model weights.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

### 5. Final Evaluation and Results
Finally, we evaluate the model on the unseen test set. We generate a **Classification Report** to see Precision, Recall, and F1-Score, which are more important than accuracy for imbalanced fraud data.

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)

print("Final Model Performance Report:")
print(classification_report(y_test, predicted))